In [1]:
!pip install playwright

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 MB 9.3 MB/s eta 0:00:00


In [2]:
!playwright install

164.9 MiB [] 0% 10.9s164.9 MiB [] 0% 13.8s164.9 MiB [] 0% 9.6s164.9 MiB [] 0% 7.7s164.9 MiB [] 0% 6.6s164.9 MiB [] 1% 6.3s164.9 MiB [] 1% 5.5s164.9 MiB [] 2% 5.1s164.9 MiB [] 2% 4.9s164.9 MiB [] 3% 4.7s164.9 MiB [] 3% 4.3s164.9 MiB [] 4% 4.1s164.9 MiB [] 4% 4.0s164.9 MiB [] 5% 3.9s164.9 MiB [] 5% 3.8s164.9 MiB [] 6% 3.8s164.9 MiB [] 6% 4.1s164.9 MiB [] 6% 4.4s164.9 MiB [] 6% 4.3s164.9 MiB [] 7% 4.2s164.9 MiB [] 7% 4.3s164.9 MiB [] 7% 4.7s164.9 MiB [] 7% 4.8s164.9 MiB [] 8% 4.7s164.9 MiB [] 8% 4.6s164.9 MiB [] 9% 4.6s164.9 MiB [] 9% 4.7s164.9 MiB [] 9% 4.6s164.9 MiB [] 10% 4.6s164.9 MiB [] 10% 4.4s164.9 MiB [] 10% 4.6s164.9 MiB [] 11% 4.5s164.9 MiB [] 11% 4.6s164.9 MiB [] 12% 4.6s164.9 MiB [] 12% 4.7s164.9 MiB [] 12% 4.5s164.9 MiB [] 13% 4.4s164.9 MiB [] 13% 4.3s164.9 MiB [] 14% 4.2s164.9 MiB [] 15% 4.1s164.9 MiB [] 15% 4.0s164.9 MiB [] 16% 3.9s164.9 MiB [] 17% 3.7s164.9 MiB [] 18% 3.6s164.9 MiB [] 19% 3.6s164.9 MiB [] 20% 3.6s164.9 MiB [] 21% 3.6s164.9 MiB [] 21% 3.7s164.9 MiB [] 21% 3

In [5]:
from playwright.async_api import async_playwright
import asyncio
import pandas as pd

URL = "https://napolke.ru/catalog/dlya_kofeyni/kofe"

async def get_data():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=True)
        page = await browser.new_page()
        await page.goto(URL)
        await page.wait_for_selector("div.desktop-product-card-root")
        cards = await page.query_selector_all("div.desktop-product-card-root")
        product_data = []
        for card in cards:
            name = await card.query_selector("a.product-title")
            name = await name.inner_text()
            link = await card.query_selector("a.product-title")
            link = await link.get_attribute("href")
            price = await card.query_selector("span.int-price")
            price = await price.inner_text()
            unit = await card.query_selector("div.wdio-card-unit")
            unit = await unit.inner_text()
            delivery_date = await card.query_selector("div.wdio-card-delivery-date")
            delivery_date = await delivery_date.inner_text()
            product_data.append({
                "Название": name,
                "Ссылка": link,
                "Цена": price,
                "Единица измерения": unit,
                "Дата доставки": delivery_date
            })
        df = pd.DataFrame(product_data)
        df.to_csv("product_data.csv", index=False, encoding="utf-8")
        await browser.close()

await get_data()

In [15]:
import re

database = pd.read_csv("product_data.csv")

def extract_weight(text): # вот эту функцию подсказал гпт, потому что я не понимала как вырезать конкретный кусок текста из строки. В функции разобралась прежде, чем вставить в код (не баньте, пожалуйста)
  match = re.search(r"(\d+\.?\d*)\s*(гр|кг)", text)
  weight = float(match.group(1))
  unit = match.group(2)
  if unit == "кг":
    weight *= 1000
  return int(weight)

database["Вес (гр)"] = database["Название"].apply(extract_weight)
price_mean = database["Цена"].mean()
weight_mean = database["Вес (гр)"].mean()
database["Цена за грамм"] = database["Цена"] / database["Вес (гр)"]
display(database)
print("Средний вес пачки кофе от Эвотор:", weight_mean)
print("Средняя цена пачки кофе от Эвотор:", price_mean)
print("Средневзешенная цена грамма кофе Эвотор:", price_mean/weight_mean)

,Название,Ссылка,Цена,Единица измерения,Дата доставки,Вес (гр),Цена за грамм
0,Кофе Jardin Dessert молотый натуральный жарены...,/catalog/chay_kofe_kakao/molotyy_kofe/product/...,403,12 шт в упаковке,26 марта\nПродоптимус,250,1.612000
1,"Кофе LavAzza Oro молотый 250 гр., флоу-пак",/catalog/chay_kofe_kakao/molotyy_kofe/product/...,498,1 шт в упаковке,Послезавтра\nЭлиза,250,1.992000
2,"Кофе в зернах Espresso Forte, Piazza del Caffe...",/catalog/chay_kofe_kakao/zernovoy_kofe/product...,1216,6 шт в упаковке,26 марта\nПродоптимус,1000,1.216000
3,Кофе Lebo Extra молотый для турки 100 гр,/catalog/chay_kofe_kakao/molotyy_kofe/product/...,173,1 шт в упаковке,Послезавтра\nРеноме,100,1.730000
4,"Кофе Жокей по-восточному молотый 100 гр., в/у",/catalog/chay_kofe_kakao/molotyy_kofe/product/...,146,18 шт в упаковке,26 марта\nПродоптимус,100,1.460000
5,"Кофе в зернах Julius Meinl Юбилейный, 500 гр.,...",/catalog/chay_kofe_kakao/zernovoy_kofe/product...,1246,3 шт в упаковке,25 марта\nTuttoFood_Pro,500,2.492000
6,"Кофе молотый ХОРС Бушидо Red Katana, 227 гр., ...",/catalog/chay_kofe_kakao/molotyy_kofe/product/...,571,1 шт в упаковке,Послезавтра\nЭлиза,227,2.515419
7,Кофе Lavazza Espresso натуральный жареный моло...,/catalog/chay_kofe_kakao/zernovoy_kofe/product...,752,3 шт в упаковке,25 марта\nTuttoFood_Pro,250,3.008000
8,"Кофе в зернах Lavazza Qualita Oro, 250 гр., фл...",/catalog/chay_kofe_kakao/zernovoy_kofe/product...,509,1 шт в упаковке,Послезавтра\nЭлиза,250,2.036000
9,"Кофе в зернах Jardin Espresso di Milano 1 кг.,...",/catalog/chay_kofe_kakao/zernovoy_kofe/product...,1484,6 шт в упаковке,26 марта\nПродоптимус,1000,1.484000


Средний вес пачки кофе от Эвотор: 537.5714285714286
Средняя цена пачки кофе от Эвотор: 888.0357142857143
Средневзешенная цена грамма кофе Эвотор: 1.6519399415360085


In [16]:
import plotly.express as px

fig = px.bar(
    database,
    x="Название",
    y="Цена за грамм",
    text="Цена за грамм",
    title="Уровень цен на кофе поставщика Эвотор",
    labels={"Средневзвешенная цена видов кофе в Эвотор": "Цена за грамм", "Название": "Название кофе"},
    color="Цена за грамм",
    color_continuous_scale="purples",
)
fig.update_traces(texttemplate='%{text:.2f} ₽', textposition="outside")
fig.update_layout(
    xaxis_title="Название кофе",
    yaxis_title="Цена за грамм",
    template="plotly_white",
    width=1000,
    height=1000,
    xaxis_tickangle=-45
)
fig.show()